In [22]:
import pandas as pd

df = pd.read_csv("../data/sorular.csv")

print("Toplam soru sayısı:", len(df))
df.head()

Toplam soru sayısı: 15


,id,language,type,question,reference_answer,evidence_section,evidence_snippet
0,1,en,A,How long does one side of the record play?,About an hour (between 53 and 54 minutes).,Playback,between 53 and 54 minutes
1,2,en,A,What is the record made of?,"A copper disk 12 inches (30 cm) in diameter, p...",Manufacturing,copper disk 12 inches
2,3,en,A,How many images are on the record?,116 images (one of them used for calibration).,Contents,116 images (one used for calibration)
3,4,en,A,In how many languages are there spoken greetin...,55 ancient and modern languages.,Contents,55 ancient and modern languages
4,5,en,A,"Who served as sound engineer for the project, ...","Jimmy Iovine, on the recommendation of John Le...",Contents,at the recommendation of John Lennon


In [23]:
import ollama

MODEL_NAME = "qwen3:4b-instruct"

options = {
    "temperature": 0,
    "num_predict": 512,
    "num_ctx": 4096
}

results = []

for index, row in df.iterrows():

    question = row["question"]
    reference_answer = row["reference_answer"]

    response = ollama.chat(
        model=MODEL_NAME,
        messages=[
            {
                "role": "user",
                "content": question
            }
        ],
        options=options
    )

    answer_no_rag = response["message"]["content"]

    # Ollama süreleri nanosaniye olarak gelir
    total_duration = response["total_duration"] / 1_000_000_000
    load_duration = response["load_duration"] / 1_000_000_000
    prompt_eval_duration = response["prompt_eval_duration"] / 1_000_000_000
    eval_duration = response["eval_duration"] / 1_000_000_000

    # Model yüklenme süresini çıkar
    processing_time = total_duration - load_duration

    # Token sayıları
    prompt_tokens = response["prompt_eval_count"]
    answer_tokens = response["eval_count"]

    # Cevap üretim hızı
    if eval_duration > 0:
        tokens_per_second = answer_tokens / eval_duration
    else:
        tokens_per_second = None

    results.append({
        "id": row["id"],
        "question": question,
        "reference_answer": reference_answer,
        "answer_no_rag": answer_no_rag,

        "model": MODEL_NAME,
        "temperature": options["temperature"],
        "num_predict": options["num_predict"],
        "num_ctx": options["num_ctx"],

        "total_duration_seconds": round(total_duration, 3),
        "load_duration_seconds": round(load_duration, 3),
        "processing_time_seconds": round(processing_time, 3),

        "prompt_eval_duration_seconds": round(
            prompt_eval_duration, 3
        ),

        "eval_duration_seconds": round(
            eval_duration, 3
        ),

        "prompt_tokens": prompt_tokens,
        "answer_tokens": answer_tokens,

        "tokens_per_second": (
            round(tokens_per_second, 2)
            if tokens_per_second is not None
            else None
        )
    })

    print(
        f"{index + 1}/{len(df)} tamamlandı | "
        f"Süre: {processing_time:.2f} sn | "
        f"Prompt: {prompt_tokens} token | "
        f"Cevap: {answer_tokens} token | "
        f"Hız: {tokens_per_second:.2f} token/sn"
    )

1/15 tamamlandı | Süre: 55.87 sn | Prompt: 18 token | Cevap: 258 token | Hız: 4.72 token/sn
2/15 tamamlandı | Süre: 62.78 sn | Prompt: 15 token | Cevap: 325 token | Hız: 5.25 token/sn
3/15 tamamlandı | Süre: 4.08 sn | Prompt: 16 token | Cevap: 25 token | Hız: 6.46 token/sn
4/15 tamamlandı | Süre: 30.74 sn | Prompt: 20 token | Cevap: 181 token | Hız: 6.12 token/sn
5/15 tamamlandı | Süre: 20.91 sn | Prompt: 22 token | Cevap: 119 token | Hız: 6.01 token/sn
6/15 tamamlandı | Süre: 27.45 sn | Prompt: 22 token | Cevap: 157 token | Hız: 5.98 token/sn
7/15 tamamlandı | Süre: 48.29 sn | Prompt: 18 token | Cevap: 270 token | Hız: 5.69 token/sn
8/15 tamamlandı | Süre: 13.04 sn | Prompt: 23 token | Cevap: 72 token | Hız: 6.05 token/sn
9/15 tamamlandı | Süre: 72.51 sn | Prompt: 32 token | Cevap: 393 token | Hız: 5.51 token/sn
10/15 tamamlandı | Süre: 67.67 sn | Prompt: 21 token | Cevap: 379 token | Hız: 5.70 token/sn
11/15 tamamlandı | Süre: 20.74 sn | Prompt: 26 token | Cevap: 117 token | Hız: 6.0

In [25]:
import os

results_df = pd.DataFrame(results)

os.makedirs(
    "../results/rag_vs_no_rag",
    exist_ok=True
)

file_path = "../results/rag_vs_no_rag/no_rag_results.csv"

results_df.to_csv(
    file_path,
    index=False,
    encoding="utf-8-sig"
)

print("Sonuçlar kaydedildi:", file_path)
print("Kaydedilen soru sayısı:", len(results_df))

results_df

Sonuçlar kaydedildi: ../results/rag_vs_no_rag/no_rag_results.csv
Kaydedilen soru sayısı: 15


,id,question,reference_answer,answer_no_rag,model,temperature,num_predict,num_ctx,total_duration_seconds,load_duration_seconds,processing_time_seconds,prompt_eval_duration_seconds,eval_duration_seconds,prompt_tokens,answer_tokens,tokens_per_second
0,1,How long does one side of the record play?,About an hour (between 53 and 54 minutes).,The length of time one side of a record plays ...,qwen3:4b-instruct,0,512,4096,55.888,0.021,55.867,1.124,54.642,18,258,4.72
1,2,What is the record made of?,"A copper disk 12 inches (30 cm) in diameter, p...","The term ""record"" can refer to different thing...",qwen3:4b-instruct,0,512,4096,62.801,0.018,62.783,0.803,61.943,15,325,5.25
2,3,How many images are on the record?,116 images (one of them used for calibration).,"I don't know what you mean by ""record"". If you...",qwen3:4b-instruct,0,512,4096,4.091,0.014,4.077,0.162,3.872,16,25,6.46
3,4,In how many languages are there spoken greetin...,55 ancient and modern languages.,"As of now, there is no official record or wide...",qwen3:4b-instruct,0,512,4096,30.756,0.016,30.740,1.112,29.578,20,181,6.12
4,5,"Who served as sound engineer for the project, ...","Jimmy Iovine, on the recommendation of John Le...","The question ""Who served as sound engineer for...",qwen3:4b-instruct,0,512,4096,20.928,0.015,20.913,1.078,19.794,22,119,6.01
5,6,What budget did John Casani give Carl Sagan fo...,"$1,500 for six weeks of work.",There is no historical record or credible sour...,qwen3:4b-instruct,0,512,4096,27.467,0.017,27.449,1.176,26.233,22,157,5.98
6,7,Who provided the blank records for the Golden ...,"Pyral S.A. of Créteil, France.",The blank records for the Golden Record were p...,qwen3:4b-instruct,0,512,4096,48.305,0.019,48.286,0.795,47.451,18,270,5.69
7,8,What award did the 40th anniversary edition of...,A Grammy Award for best boxed or limited-editi...,The 40th anniversary edition of the record did...,qwen3:4b-instruct,0,512,4096,13.050,0.012,13.038,1.109,11.896,23,72,6.05
8,9,What is the half-life of the uranium-238 on th...,4.468 billion years. A civilization that finds...,"Uranium-238 does not have a ""half-life"" listed...",qwen3:4b-instruct,0,512,4096,72.530,0.016,72.515,1.159,71.317,32,393,5.51
9,10,What does the Golden Record have in common wit...,The pulsar map and the hydrogen molecule diagr...,The **Golden Record** and the **Pioneer plaque...,qwen3:4b-instruct,0,512,4096,67.685,0.014,67.671,1.108,66.514,21,379,5.70


In [27]:
print("NO-RAG EXPERIMENT SUMMARY")
print("-------------------------")

print("Model:", MODEL_NAME)
print("Temperature:", options["temperature"])
print("Num Predict:", options["num_predict"])
print("Num Context:", options["num_ctx"])

print(
    "Ortalama işlem süresi:",
    round(results_df["processing_time_seconds"].mean(), 3),
    "saniye"
)

print(
    "Ortalama prompt token:",
    round(results_df["prompt_tokens"].mean(), 2)
)

print(
    "Ortalama cevap token:",
    round(results_df["answer_tokens"].mean(), 2)
)

print(
    "Ortalama token/s:",
    round(results_df["tokens_per_second"].mean(), 2)
)

NO-RAG EXPERIMENT SUMMARY
-------------------------
Model: qwen3:4b-instruct
Temperature: 0
Num Predict: 512
Num Context: 4096
Ortalama işlem süresi: 37.553 saniye
Ortalama prompt token: 21.33
Ortalama cevap token: 198.6
Ortalama token/s: 5.62
